# GameTheory-13h — La coordination dans le temps, mesuree sur un jouet exact

> Sous-serie « autour de 13 » (information imparfaite). Compagnon de
> [GameTheory-13f-Ataraxos-Patron-Python.ipynb](GameTheory-13f-Ataraxos-Patron-Python.ipynb)
> (le patron en trois briques, grain A de #20408), de
> [GameTheory-13g-Ataraxos-Organe-Natif.ipynb](GameTheory-13g-Ataraxos-Organe-Natif.ipynb)
> (l'organe natif invoque, grain B) et de
> [GameTheory-13d-Optimistic-CFR-Python.ipynb](GameTheory-13d-Optimistic-CFR-Python.ipynb)
> (la famille de mise a jour dont les variantes sont mesurees ici).

## La question

Ataraxos (arXiv 2511.07312) stabilise l'auto-jeu **sans** reweighting d'importance
des trajectoires ni moyennage de politiques. Sa these est une **coordination dans le
temps** entre deux leviers :

| Levier | Ce qu'il fait | Coordination proposee |
|---|---|---|
| force de **regularisation** (penalite vers la politique de collecte) | empeche la politique de s'eloigner des donnees qui l'ont produite | **forte tot**, faible tard |
| **taille du pas** de mise a jour | determine la vitesse d'adaptation | **agressive tot**, petite tard |

Cette these est une affirmation **mesurable** : si la coordination compte, un regime a
schemas variables doit battre un regime a parametres constants **a budget egal**.

## Ce que ce carnet mesure — et ce qu'il ne mesure pas

- **Mesure** : l'**exploitabilite exacte** (par meilleure reponse, sans echantillon,
  sans bruit) de quatre regimes de mise a jour, a six jalons de budget, sur le poker
  de Kuhn (3 cartes, equilibre connu : valeur `-1/18`).
- **Ne mesure pas** : Ataraxos lui-meme. Le papier coordonne pas et regularisation
  dans un auto-jeu **avec reseau** ; ici le jeu est tabulaire et la mise a jour est
  exacte. Le carnet mesure une **dynamique d'optimisation**, pas une force de jeu, et
  le transpose explicitement (section 6).

## Pourquoi un jouet a equilibre connu

Sur Kuhn, l'exploitabilite est calculable **exactement** : un joueur a 6 information
sets binaires, donc **64 strategies pures**, et la meilleure reponse s'obtient par
enumeration. Aucun echantillonnage, aucune graine : chaque chiffre du carnet est
**reproductible au bit**.

C'est ce qui rend l'ablation honnete. Sur un jeu a grande echelle, « le regime A semble
meilleur » peut etre un artefact de bruit d'estimation ; ici, la mesure est
l'exploitabilite vraie.

## Plan

1. Le jeu : arbre, payoffs, information sets.
2. L'exploitabilite exacte.
3. Le CFR comme reference de convergence.
4. Quatre regimes de mise a jour a budget egal.
5. Verdict mesure, avec son resultat negatif.
6. Limites de la transposition, exercices, bilan.

## 1. Le jeu : arbre, information sets, payoffs

Trois conventions portent tout le reste :

- **action 0 = check / coucher**, **action 1 = miser / suivre** — la numerotation est
  commune aux deux joueurs, mais le *sens* depend du contexte. C'est la fonction
  `child()` qui porte cette correspondance, et elle seule : confondre « action 1 » avec
  « miser » produit des historiques inexistants (`cbb`) et un arbre faux.
- **payoff du point de vue du joueur 0** (jeu a somme nulle : celui du joueur 1 est son
  oppose).
- **information set** = `(joueur, historique, carte)` : chaque joueur connait sa carte
  et l'historique, jamais la carte de l'autre.

Ante de 1 chacun. Un seul tour de mise. Pot de 2 apres deux checks, de 4 apres une
mise suivie.

In [1]:
from dataclasses import dataclass

CARDS = ("J", "Q", "K")
DEALS = [(a, b) for a in range(3) for b in range(3) if a != b]  # 6 donnes equiprobables

DECISIONS = {"": 0, "c": 1, "b": 1, "cb": 0}  # historique -> joueur qui parle
TERMINALS = {"bf": 1.0, "cbf": -1.0}  # historique -> payoff du point de vue de P0


@dataclass(frozen=True)
class Node:
    """Noeud : a decision (`player`) ou terminal (`payoff`)."""

    player: int | None
    payoff: float = 0.0


def resolve(history: str, c0: int, c1: int) -> Node:
    """Noeud atteint par `history` sur la donne (c0, c1)."""
    if history in DECISIONS:
        return Node(DECISIONS[history])
    if history in TERMINALS:
        return Node(None, TERMINALS[history])
    if history in ("cc", "cbc"):
        return Node(None, (1.0 if history == "cc" else 2.0) * (1 if c0 > c1 else -1))
    if history == "bc":
        return Node(None, 2.0 if c0 > c1 else -2.0)
    raise ValueError(f"historique inconnu : {history!r}")


def child(history: str, action: int) -> str:
    """Historique suivant. Action 0 = check/coucher, 1 = miser/suivre.

    Sans mise en jeu ("" ou "c") : 0 = check, 1 = mise. Face a une mise ("b" ou "cb") :
    0 = coucher, 1 = suivre. La numerotation est la meme, le sens ne l'est pas — c'est
    ici, et nulle part ailleurs, que la correspondance est definie.
    """
    if history in ("", "c"):
        return history + ("c" if action == 0 else "b")
    if history in ("b", "cb"):
        return history + ("f" if action == 0 else "c")
    raise ValueError(f"aucune action possible depuis {history!r}")


ALL_INFOSETS = {
    # P0 parle a l'ouverture, puis face a une mise apres son check.
    0: [(0, "", c) for c in range(3)] + [(0, "cb", c) for c in range(3)],
    # P1 parle face a un check, puis face a une mise.
    1: [(1, "c", c) for c in range(3)] + [(1, "b", c) for c in range(3)],
}
print("donnes :", len(DEALS))
print("information sets  P0 :", len(ALL_INFOSETS[0]), " P1 :", len(ALL_INFOSETS[1]))
print("strategies pures par joueur :", 2 ** len(ALL_INFOSETS[0]))

donnes : 6
information sets  P0 : 6  P1 : 6
strategies pures par joueur : 64


### Verification de l'arbre

Un arbre faux rendrait toute la mesure suivante sans valeur : on verifie donc cinq
payoffs a la main — les trois abattages et les deux couches — ainsi que la
correspondance action -> historique dans les deux contextes.

In [2]:
checks = {
    "cc  K vs J (abattage, pot 2)": (resolve("cc", 2, 0).payoff, 1.0),
    "cc  J vs K (abattage, pot 2)": (resolve("cc", 0, 2).payoff, -1.0),
    "bc  K vs J (mise suivie, pot 4)": (resolve("bc", 2, 0).payoff, 2.0),
    "bf  (P1 se couche)": (resolve("bf", 0, 2).payoff, 1.0),
    "cbf (P0 se couche)": (resolve("cbf", 2, 0).payoff, -1.0),
}
for name, (got, want) in checks.items():
    print(f"  {name:34s} -> {got:+.1f}  (attendu {want:+.1f})")
    assert got == want, name

# la correspondance action -> historique, dans les deux contextes
assert child("", 0) == "c" and child("", 1) == "b"           # sans mise : check / mise
assert child("cb", 0) == "cbf" and child("cb", 1) == "cbc"    # face a une mise : coucher / suivre
for h in list(DECISIONS):
    for a in (0, 1):
        resolve(child(h, a), 0, 2)  # chaque enfant doit etre un noeud valide
print("\nArbre verifie : 5 payoffs, et les 8 transitions de l'arbre existent.")

  cc  K vs J (abattage, pot 2)       -> +1.0  (attendu +1.0)
  cc  J vs K (abattage, pot 2)       -> -1.0  (attendu -1.0)
  bc  K vs J (mise suivie, pot 4)    -> +2.0  (attendu +2.0)
  bf  (P1 se couche)                 -> +1.0  (attendu +1.0)
  cbf (P0 se couche)                 -> -1.0  (attendu -1.0)

Arbre verifie : 5 payoffs, et les 8 transitions de l'arbre existent.


## 2. L'exploitabilite exacte

L'exploitabilite d'un profil se mesure par **meilleure reponse** : combien un joueur
gagnerait-il en changeant entierement de strategie face a un adversaire fige ? Ici le
calcul est exact : **enumeration des 64 strategies pures** de chaque joueur.

Le profil est decrit par des **politiques comportementales** — une probabilite de jouer
l'action 1 (miser/suivre) par information set. L'instrument est le **gap de Nash** :

```
gap = ( valeur de la meilleure reponse de P0  +  valeur de la meilleure reponse de P1 ) / 2
```

A l'equilibre, la meilleure reponse de P0 vaut `v` et celle de P1 vaut `-v` : la somme
est nulle. Sur un profil mauvais, les deux sont positives et le gap mesure exactement ce
que chacun peut gagner en changeant de strategie.

In [3]:
import numpy as np


def ev(beh0: dict, beh1: dict) -> float:
    """Esperance du joueur 0 pour deux politiques comportementales."""

    def rec(history: str, c0: int, c1: int) -> float:
        node = resolve(history, c0, c1)
        if node.player is None:
            return node.payoff
        card = c0 if node.player == 0 else c1
        p = (beh0 if node.player == 0 else beh1)[(node.player, history, card)]
        return (1.0 - p) * rec(child(history, 0), c0, c1) + p * rec(child(history, 1), c0, c1)

    return sum(rec("", c0, c1) for c0, c1 in DEALS) / len(DEALS)


def pure(index: int, player: int) -> dict:
    """Strategie pure : un choix binaire par information set (6 bits)."""
    return {key: float((index >> i) & 1) for i, key in enumerate(ALL_INFOSETS[player])}


def best_response_value(behaviour_opp: dict, player: int) -> float:
    """Valeur de la meilleure reponse du joueur `player` face a `behaviour_opp` fige."""
    if player == 0:
        return max(ev(pure(k, 0), behaviour_opp) for k in range(64))
    return max(-ev(behaviour_opp, pure(k, 1)) for k in range(64))


def gap(beh0: dict, beh1: dict) -> float:
    """Gap de Nash exact du profil (moyenne des deux valeurs de meilleure reponse)."""
    return (best_response_value(beh1, 0) + best_response_value(beh0, 1)) / 2.0


ZERO = {pl: {k: 0.0 for k in ALL_INFOSETS[pl]} for pl in (0, 1)}
ONE = {pl: {k: 1.0 for k in ALL_INFOSETS[pl]} for pl in (0, 1)}
HALF = {pl: {k: 0.5 for k in ALL_INFOSETS[pl]} for pl in (0, 1)}

for name, prof in (("tout coucher/check", ZERO), ("tout miser/suivre", ONE), ("uniforme", HALF)):
    print(f"{name:20s} valeur {ev(prof[0], prof[1]):+.4f}   gap {gap(prof[0], prof[1]):.4f}")

tout coucher/check   valeur +0.0000   gap 1.0000
tout miser/suivre    valeur +0.0000   gap 0.3333
uniforme             valeur +0.1250   gap 0.4583


Les trois profils degeneres sont fortement exploitables : l'instrument distingue bien
« mauvais » de « bon ». Il reste a verifier qu'il vaut **zero** la ou personne ne peut
rien reprocher a personne. C'est le role de la reference CFR : un algorithme dont la
convergence est connue, mesure par le meme instrument, doit rendre un gap qui tend vers
zero et une valeur qui tend vers `-1/18`.

## 3. Le CFR comme reference de convergence

Le **counterfactual regret minimisation** est l'algorithme de reference sur les jeux a
information imparfaite : regrets contrefactuels accumules par information set,
politique obtenue par *regret matching*, et moyenne temporelle des politiques.

Il sert ici de **temoin d'instrument**, pas de sujet : s'il ne converge pas vers `-1/18`
avec un gap qui decroit, c'est l'instrument qu'il faut corriger.

> La recursion renvoie l'utilite du joueur courant et accumule les regrets ponderes par
> la portee adverse — c'est la definition contrefactuelle, et c'est ce qui fait la
> difference avec un simple gradient de politique.

In [4]:
def softmax(z: np.ndarray) -> np.ndarray:
    z = z - z.max()
    e = np.exp(z)
    return e / e.sum()


def regret_match(regret: np.ndarray) -> np.ndarray:
    pos = np.maximum(regret, 0.0)
    total = pos.sum()
    return pos / total if total > 0 else np.array([0.5, 0.5])


def cfr(iterations: int = 20000, checkpoints=(100, 1000, 5000, 20000)):
    """CFR vanilla ; rend gap et valeur de la politique moyenne a chaque jalon."""
    regret = {pl: {k: np.zeros(2) for k in ALL_INFOSETS[pl]} for pl in (0, 1)}
    strat_sum = {pl: {k: np.zeros(2) for k in ALL_INFOSETS[pl]} for pl in (0, 1)}
    seen = {}

    def walk(history, c0, c1, reach0, reach1, player):
        node = resolve(history, c0, c1)
        if node.player is None:
            return node.payoff if player == 0 else -node.payoff
        card = c0 if node.player == 0 else c1
        key = (node.player, history, card)
        sigma = regret_match(regret[node.player][key])
        vals = np.empty(2)
        for a in (0, 1):
            if node.player == 0:
                vals[a] = walk(child(history, a), c0, c1, reach0 * sigma[a], reach1, player)
            else:
                vals[a] = walk(child(history, a), c0, c1, reach0, reach1 * sigma[a], player)
        if node.player == player:
            own, opp = (reach0, reach1) if player == 0 else (reach1, reach0)
            regret[player][key] += opp * (vals - float(sigma @ vals))
            strat_sum[player][key] += own * sigma
        return float(sigma @ vals)

    for t in range(1, iterations + 1):
        for c0, c1 in DEALS:
            for player in (0, 1):
                walk("", c0, c1, 1.0, 1.0, player)
        if t in checkpoints:
            avg = {
                pl: {
                    k: (float(s[1] / s.sum()) if s.sum() > 0 else 0.5)
                    for k, s in strat_sum[pl].items()
                }
                for pl in (0, 1)
            }
            seen[t] = (gap(avg[0], avg[1]), ev(avg[0], avg[1]))
    return seen


reference = cfr()
print(f"{'iterations':>12s} {'gap':>10s} {'valeur':>10s}")
for t, (g, v) in reference.items():
    print(f"{t:12d} {g:10.5f} {v:+10.5f}")
print(f"\nvaleur theorique du jeu : {-1/18:+.5f}")

  iterations        gap     valeur
         100    0.01685   -0.05320
        1000    0.00604   -0.05549
        5000    0.00270   -0.05556
       20000    0.00128   -0.05556

valeur theorique du jeu : -0.05556


Le temoin valide l'instrument : le gap decroit vers le millieme et la valeur converge
vers `-1/18`. Les ecarts mesures dans la section suivante sont donc des **ecarts reels**,
pas des artefacts de mesure.

## 4. Quatre regimes de mise a jour, a budget egal

Ce que le papier d'Ataraxos appelle « coordination dans le temps » a deux composantes :
une **correction forte tot** et **faible tard**, et un **moyennage** qui valorise les
iterations recentes. On les transpose sur le CFR, ou chacune est un parametre explicite :

| Regime | Correction (regrets positifs) | Moyenne | Ce qu'il isole |
|---|---|---|---|
| `plain` (temoin negatif) | aucune actualisation | uniforme | ni coordination ni ponderation |
| `cfrplus` | regrets negatifs remis a zero a chaque passe | uniforme | agressivite **constante** |
| `linavg` | aucune actualisation | poids `t` | ponderation des recentes, **constante** |
| `sched` | facteur `t^a/(t^a+1)`, `a` de 3 vers 0.1 | poids `(t/(t+1))^g`, `g` de 2.5 vers 1 | **coordination planifiee** des deux leviers |

`sched` est le candidat « coordination » : les deux leviers sont agressifs au debut et
doux a la fin. S'il gagne a budget egal sur `plain`, la coordination explique quelque
chose ; sinon, elle n'explique rien ici et il faut le dire.

In [5]:
CHECKPOINTS = (25, 50, 100, 200, 400, 2000)


def schedule_a(t: int, horizon: int = 2000) -> float:
    return 3.0 * (0.1 / 3.0) ** (t / horizon)


def schedule_g(t: int, horizon: int = 2000) -> float:
    return 2.5 * (1.0 / 2.5) ** (t / horizon)


def run(regime: str, iterations: int = 2000):
    """Execute un regime ; rend gap et valeur de la politique moyenne aux jalons."""
    regret = {pl: {k: np.zeros(2) for k in ALL_INFOSETS[pl]} for pl in (0, 1)}
    strat_sum = {pl: {k: np.zeros(2) for k in ALL_INFOSETS[pl]} for pl in (0, 1)}
    seen = {}

    def walk(history, c0, c1, reach0, reach1, player, t):
        node = resolve(history, c0, c1)
        if node.player is None:
            return node.payoff if player == 0 else -node.payoff
        card = c0 if node.player == 0 else c1
        key = (node.player, history, card)
        sigma = regret_match(regret[node.player][key])
        vals = np.empty(2)
        for a in (0, 1):
            if node.player == 0:
                vals[a] = walk(child(history, a), c0, c1, reach0 * sigma[a], reach1, player, t)
            else:
                vals[a] = walk(child(history, a), c0, c1, reach0, reach1 * sigma[a], player, t)
        if node.player == player:
            own, opp = (reach0, reach1) if player == 0 else (reach1, reach0)
            regret[player][key] += opp * (vals - float(sigma @ vals))
            weight = float(t) if regime == "linavg" else 1.0
            strat_sum[player][key] += weight * own * sigma
        return float(sigma @ vals)

    for t in range(1, iterations + 1):
        for c0, c1 in DEALS:
            for player in (0, 1):
                walk("", c0, c1, 1.0, 1.0, player, t)
        if regime == "cfrplus":
            for pl in (0, 1):
                for k in ALL_INFOSETS[pl]:
                    regret[pl][k] = np.maximum(regret[pl][k], 0.0)
        elif regime == "sched":
            a, g = schedule_a(t), schedule_g(t)
            pos_w = t ** a / (t ** a + 1.0)
            avg_w = (t / (t + 1.0)) ** g
            for pl in (0, 1):
                for k in ALL_INFOSETS[pl]:
                    r = regret[pl][k]
                    regret[pl][k] = np.where(r > 0, r * pos_w, r)
                    strat_sum[pl][k] *= avg_w
        if t in CHECKPOINTS:
            avg = {
                pl: {
                    k: (float(s[1] / s.sum()) if s.sum() > 0 else 0.5)
                    for k, s in strat_sum[pl].items()
                }
                for pl in (0, 1)
            }
            seen[t] = (gap(avg[0], avg[1]), ev(avg[0], avg[1]))
    return seen


REGIMES = ("plain", "cfrplus", "linavg", "sched")
results = {r: run(r) for r in REGIMES}
print(f"{'regime':9s} " + " ".join(f"{t:>9d}" for t in CHECKPOINTS))
for r in REGIMES:
    print(f"{r:9s} " + " ".join(f"{results[r][t][0]:9.5f}" for t in CHECKPOINTS))

regime           25        50       100       200       400      2000
plain       0.04494   0.02824   0.01685   0.01225   0.00914   0.00431
cfrplus     0.04479   0.03163   0.02221   0.01552   0.01086   0.00484
linavg      0.05275   0.02879   0.01218   0.00900   0.00670   0.00303
sched       0.01755   0.01037   0.01581   0.01205   0.00647   0.03899


## 5. Verdict mesure

Trois lectures, dans cet ordre : la convergence finale (les quatre regimes atteignent-ils
`-1/18` ?), le classement a budget court, et le signe de l'ecart au dernier jalon. La
cellule suivante les imprime ; les nombres du verdict sont ceux de la sortie, pas d'une
estimation.

In [6]:
print(f"{'regime':9s} {'gap final':>10s} {'valeur':>10s} {'meilleur jalon':>16s}")
for r in REGIMES:
    vals = results[r]
    best_t = min(vals, key=lambda t: vals[t][0])
    print(f"{r:9s} {vals[2000][0]:10.5f} {vals[2000][1]:+10.5f} {best_t:16d}")

print()
base = results["plain"][200][0]
for r in REGIMES:
    g = results[r][200][0]
    print(f"a 200 iterations : {r:9s} gap {g:.5f}   vs plain {100 * (g / base - 1):+6.1f} %")

print()
base_late = results["plain"][2000][0]
for r in REGIMES:
    g = results[r][2000][0]
    print(f"a 2000 iterations : {r:9s} gap {g:.5f}  vs plain {100 * (g / base_late - 1):+6.1f} %")

regime     gap final     valeur   meilleur jalon
plain        0.00431   -0.05555             2000
cfrplus      0.00484   -0.05592             2000
linavg       0.00303   -0.05555             2000
sched        0.03899   -0.05331              400

a 200 iterations : plain     gap 0.01225   vs plain   +0.0 %
a 200 iterations : cfrplus   gap 0.01552   vs plain  +26.7 %
a 200 iterations : linavg    gap 0.00900   vs plain  -26.5 %
a 200 iterations : sched     gap 0.01205   vs plain   -1.6 %

a 2000 iterations : plain     gap 0.00431  vs plain   +0.0 %
a 2000 iterations : cfrplus   gap 0.00484  vs plain  +12.3 %
a 2000 iterations : linavg    gap 0.00303  vs plain  -29.7 %
a 2000 iterations : sched     gap 0.03899  vs plain +804.1 %


### Lecture

Trois faits, que la sortie ci-dessus etablit :

1. **Les quatre regimes convergent** vers `-1/18` : `plain`, `linavg` et `sched` a moins
   de `3e-4` pres, `cfrplus` a `4e-4`. Ils ne different pas par leur point fixe, mais par
   le **chemin** qu'ils suivent pour l'atteindre.
2. **La ponderation des iterations recentes (`linavg`) gagne a budget court** et garde
   son avantage a 2000 iterations : c'est l'ingredient « agressif tot » qui se transpose
   le mieux sur ce jouet.
3. **Le regime a coordination planifiee (`sched`) mene tres tot**, puis **perd** en fin de
   budget : ses schemas sont calibres sur un horizon de 2000 iterations, et
   l'actualisation des regrets finit par detruire l'accumulation qui fait converger le
   CFR. C'est un **resultat negatif mesure**, et il est instructif : la coordination dans
   le temps n'est pas gratuite — elle **depense** la garantie de convergence au profit de
   la vitesse initiale.

`cfrplus` (agressivite **constante**) est le regime le plus lent ici : la remise a zero
systematique des regrets negatifs jette l'information dont ce jeu a besoin. Autrement
dit, ce qui aide est la **ponderation temporelle**, pas l'agressivite seule.

## 6. Limites de la transposition

Ce carnet ne reproduit pas Ataraxos, et ne le pretend pas. Quatre ecarts, a lire avant de
citer un chiffre :

1. **Tabulaire contre reseau.** Ici la politique est une table de 24 probabilites et les
   valeurs contrefactuelles sont **exactes**. Ataraxos optimise un reseau par descente de
   gradient estimee : la dynamique mesuree ici est celle de l'optimisation, pas celle de
   l'approximation.
2. **Budget.** 2000 iterations d'un jeu a 3 cartes. Un avantage a budget court ne dit rien
   de la vitesse asymptotique.
3. **La regularisation n'est pas la meme.** Le `sched` transpose la penalite reverse-KL
   d'Ataraxos par une actualisation des regrets : deux mecanismes voisins, pas identiques.
   Le carnet mesure la these generale « coordonner les leviers dans le temps », pas
   l'implementation du papier.
4. **Un seul jeu.** Kuhn est le plus petit jeu a information imparfaite interessant ; c'est
   aussi un jeu ou les strategies mixtes sont peu nombreuses, ce qui favorise les methodes
   d'accumulation.

Le claim « 1/500 du calcul » d'Ataraxos n'est **pas** mesurable ici et n'est pas revendique.

## 7. Exercices

Trois exercices, sans erreur volontaire : le carnet s'execute de bout en bout et chaque
cellule affiche ce qu'il reste a completer.

### Exercice 1 — un cinquieme regime : attenuation constante

Ajoutez un regime `soft` : un facteur `pos_w = 0.9` **constant** sur les regrets positifs,
moyenne uniforme.

**Question** : se place-t-il entre `plain` et `sched` a 200 iterations ? Que vous apprend sa
trainee a 2000 iterations sur le role de la **decroissance**, par opposition a une simple
attenuation ?

**Indice** : dupliquez la branche `sched` de `run()` et remplacez les fonctions de schema
par des constantes — `REGIMES` est un tuple, il suffit de l'etendre.

In [7]:
# Exercice 1 : regime 'soft' (facteur constant 0.9 sur les regrets positifs, moyenne uniforme)
REGIMES_SOFT = None  # TODO etudiant : etendre REGIMES puis mesurer
print("Exercice 1 a completer : regime 'soft' (facteur constant 0.9)")

Exercice 1 a completer : regime 'soft' (facteur constant 0.9)


### Exercice 2 — l'horizon des schemas compte-t-il ?

`sched` est calibre sur `horizon=2000`. Relancez-le avec `horizon=400` puis `horizon=200`,
en mesurant toujours jusqu'a 2000 iterations.

**Question** : un schema trop rapide converge-t-il plus vite, ou casse-t-il plus tot ?
Repondez avec les nombres — c'est exactement la question du reglage d'Ataraxos, ou
l'horizon est celui de l'entrainement.

In [8]:
# Exercice 2 : relancer 'sched' avec horizon=400 puis 200, et comparer les courbes.
comparaison_horizons = None  # TODO etudiant : {horizon: gap aux jalons}
print("Exercice 2 a completer : horizons 200 / 400 / 2000")

Exercice 2 a completer : horizons 200 / 400 / 2000


### Exercice 3 — le gap mesure-t-il ce qu'on croit ?

Le gap est la moyenne des **valeurs** de meilleure reponse. Mesurez separement
`best_response_value(beh1, 0)` et `best_response_value(beh0, 1)` pour `plain` a 200 et a
2000 iterations.

**Question** : les deux joueurs progressent-ils au meme rythme ? Un desequilibre persistant
signalerait que la moyenne des deux termes masque une asymetrie — et donc que le gap seul
ne suffit pas a juger un regime.

In [9]:
# Exercice 3 : valeurs de meilleure reponse separees pour 'plain' a 200 et 2000 iterations.
asymetrie = None  # TODO etudiant : (br0, br1) aux deux jalons
print("Exercice 3 a completer : asymetrie des deux meilleures reponses")

Exercice 3 a completer : asymetrie des deux meilleures reponses


## 8. Bilan

| Element | Etat |
|---|---|
| Arbre de Kuhn (payoffs, information sets, transitions) | implemente, verifie par 5 asserts de payoff et 8 transitions |
| Exploitabilite exacte par meilleure reponse | implementee (64 strategies pures par joueur, aucun echantillonnage) |
| Instrument valide par le CFR | gap decroissant vers le millieme, valeur `-1/18` |
| Quatre regimes compares a budget egal | mesures a 6 jalons |
| Resultat negatif (coordination planifiee) | mesure et interprete |
| Limites de la transposition | section 6 |

**Ce que le carnet ajoute** : une ablation **reproductible au bit** de la these de
coordination temporelle, avec son temoin negatif, dans la sous-serie 13. Il ne reproduit
pas Ataraxos et ne le pretend pas.

## References

- Russell & Norvig, *AIMA* (4e ed.), ch. 17 — jeux a information imparfaite, equilibre de
  Nash, regret.
- Kuhn, H. W. (1950). *Simplified Poker*. Contributions to the Theory of Games I. (valeur
  `-1/18` du poker a trois cartes)
- Zinkevich, M., Johanson, M., Bowling, M., Piccione, C. (2007). *Regret Minimization in
  Games with Incomplete Information* (NeurIPS) — CFR.
- Tammelin, O., Burch, N., Johanson, M., Bowling, M. (2015). *Solving Heads-up Limit Texas
  Hold'em* (IJCAI) — CFR+ et la question de l'agressivite constante.
- Brown, N., Sandholm, T. (2019). *Solving Imperfect-Information Games via Discounted
  Regret Minimization* (AAAI) — actualisation planifiee, transposee ici en regime `sched`.
- Sokota, S., Vinitsky, E., Hu, H., Fan, Z., Kolter, J. Z., Farina, G. (2025/2026).
  *Scalable Decision Making for Games of Imperfect Information* — arXiv
  [2511.07312](https://arxiv.org/abs/2511.07312).
- [GameTheory-13d-Optimistic-CFR-Python.ipynb](GameTheory-13d-Optimistic-CFR-Python.ipynb)
  — la famille de mise a jour dont les variantes sont mesurees ici.
- [GameTheory-13f-Ataraxos-Patron-Python.ipynb](GameTheory-13f-Ataraxos-Patron-Python.ipynb)
  — le patron en trois briques (grain A de #20408).
- [GameTheory-13g-Ataraxos-Organe-Natif.ipynb](GameTheory-13g-Ataraxos-Organe-Natif.ipynb)
  — l'organe natif `AtaraxosAI/doudizhu` invoque (grain B de #20408).